# 🌲 RandomForest com ranger em R
*Machine Learning - Ciência de Dados*

**Dataset:** Iris (versão 3 classes)
**Pacotes:** ranger, ggplot2, dplyr, vip, yardstick

In [ ]:
# Carregar bibliotecas
library(ranger)      # Random Forest
library(ggplot2)     # Visualização
library(dplyr)       # Manipulação
library(vip)         # Feature Importance
library(yardstick)   # Métricas

cat("✓ Pacotes carregados!\n")

## 1. Carregar Dados Iris

In [ ]:
df <- iris
names(df) <- c("Sepal.Length", "Sepal.Width", 
               "Petal.Length", "Petal.Width", "Species")
head(df)
str(df)

## 2. Preparar Features e Target

In [ ]:
features <- c("Sepal.Length", "Sepal.Width", 
              "Petal.Length", "Petal.Width")
X <- df[, features]
y <- as.factor(df$Species)

cat("Features:", features, "\n")
cat("Classes:", levels(y), "\n")

## 3. Divisão Treino/Teste (75/25)

In [ ]:
set.seed(42)
n <- nrow(X)
train_idx <- sample(1:n, size = 0.75 * n)

X_train <- X[train_idx, ]
X_test  <- X[-train_idx, ]
y_train <- y[train_idx]
y_test  <- y[-train_idx]

cat("Treino:", length(y_train), "amostras\n")
cat("Teste:", length(y_test), "amostras\n")

## 4. Treinar RandomForest (ranger)

In [ ]:
set.seed(42)
rf_model <- ranger(
  x = X_train, 
  y = y_train,
  num.trees = 500,
  mtry = sqrt(ncol(X)),
  importance = 'permutation'
)

print(rf_model)
cat("\nOOB Error:", round(rf_model$prediction.error * 100, 2), "%\n")

## 5. Feature Importance

In [ ]:
# Importância das features
imp <- importance(rf_model)
cat("Importance:\n")
print(sort(imp, decreasing = TRUE))

# Visualização
vip(rf_model) + ggtitle("Feature Importance - RandomForest")

## 6. Predição e Avaliação

In [ ]:
# Predizer
y_pred <- predict(rf_model, X_test)$predictions

# Matriz de confusão
cat("Matriz de Confusão:\n")
print(table(Previsto = y_pred, Real = y_test))

# Acurácia
acc <- sum(y_pred == y_test) / length(y_test)
cat("\nAcurácia:", round(acc * 100, 2), "%\n")

## 7. Validação Cruzada 5-fold

In [ ]:
set.seed(42)
folds <- sample(rep(1:5, length.out = nrow(X)))

cv_scores <- sapply(1:5, function(fold) {
  test_idx <- which(folds == fold)
  X_tr <- X[-test_idx, ]; X_te <- X[test_idx, ]
  y_tr <- y[-test_idx];   y_te <- y[test_idx]
  
  rf <- ranger(x = X_tr, y = y_tr, num.trees = 100)
  yp <- predict(rf, X_te)$predictions
  sum(yp == y_te) / length(y_te)
})

cat("CV Scores:", round(cv_scores * 100, 2), "%\n")
cat("CV Média:", round(mean(cv_scores) * 100, 2), "%\n")
cat("CV Desvio:", round(sd(cv_scores) * 100, 2), "%\n")

## 8. Conclusão

In [ ]:
cat("═══════════════════════════════════════\n")
cat("      RESULTADOS RANDOMFOREST (ranger)  \n")
cat("═══════════════════════════════════════\n")
cat("Modelo:", rf_model$mtry, "mtry,", rf_model$num.trees, "árvores\n")
cat("Acurácia (teste):", round(acc * 100, 2), "%\n")
cat("CV 5-fold média:  ", round(mean(cv_scores) * 100, 2), "%\n")
cat("OOB Error:        ", round(rf_model$prediction.error * 100, 2), "%\n")
cat("═══════════════════════════════════════\n")

cat("\nFeature mais importante:", 
    names(which.max(importance(rf_model))), "\n")